# Similarity Search and HNSW in Chroma DB

Similarity search in Chroma DB is generally straightforward, but there are a few important nuances to understand in order to use it effectively for information retrieval. Before exploring those details, it's helpful to first understand what a vector index is.

### What is a Vector Index?

Finding the most semantically similar vectors to a query is mathematically straightforward. For example, to compute cosine similarity, you normalize the embeddings of all documents and the query, then calculate the dot product between the query embedding and each document embedding. This yields the cosine similarity scores. Identifying the most similar document is simply a matter of finding the one with the highest score.

However, this brute-force approach is inefficient—it requires comparing the query against every vector in the database, which becomes slow at scale. To address this, **vector indexes** are used.

**Vector Indexes:** are specialized data structures that enable algorithms to compute similarity scores with only a small subset of vectors, significantly speeding up the search while still returning exact or near-optimal results.

The first part of the third paragraph is just re-stating what was already said — skip it. The important part is the second half, which explains **how** a vector index actually achieves that speed. Here's the key idea:

#### How a Vector Index Works Internally

A vector index doesn't store vectors as a flat list (like a plain Python list where you'd scan every item one by one). Instead it **organizes vectors spatially** — meaning it structures them based on how close they are to each other in vector space.

Two main strategies it uses:

**1) Clustering similar vectors together**
Vectors that are close to each other (semantically similar) get grouped/clustered in the index. So when a query comes in, the algorithm immediately jumps to the relevant cluster instead of scanning everything — it already knows "similar stuff lives over here."

**2) Proximity-based graphs**
Vectors get connected to their nearest neighbors via graph links (this is exactly what **HNSW** — which ChromaDB uses — does). When searching, the algorithm hops through these graph links, always moving toward more similar vectors, until it finds the best match. It never needs to look at the whole dataset.

**The Key Idea: Pruning**

Both strategies allow the algorithm to **prune** (ignore/skip) large parts of the dataset early — because the structure already tells it "the answer definitely isn't over there, don't bother looking." This is why it's fast even with millions of vectors.

**One-line summary:** a vector index is like a smart map of your vector space — instead of searching everywhere, it tells the algorithm exactly where to look.

Good questions — let me answer both clearly.

### How clustering/proximity actually gets built

They happen **automatically** when you add documents to a collection — you don't do anything manually. The moment you call `collection.add(...)`, ChromaDB's indexing engine (written in Rust, remember) automatically builds and updates the index structure in the background. You just add data and query — the index organization happens invisibly.

Think of it like this: when you called `collection.add()` with your 5 documents, ChromaDB didn't just dump them into a flat list. It organized them into the index structure automatically as part of that same operation.

#### What HNSW actually is

`HNSW (Hierarchical Navigable Small World)` is a specific **algorithm** that implements the **proximity-based graph** strategy. It's not a separate thing you choose — it's what ChromaDB uses internally by default to build and search the index.

`HNSW` is a fast, scalable graph-based vector index designed for **approximate nearest neighbor (ANN)** search in high-dimensional spaces.

**Wait... HNSW is an *algorithm* or *vector index*?**

It can be confusing, because all three terms (algorithm, data structure, index) are being used loosely to describe the same thing from different angles. But let's clarify:

- **Data structure** — how the data is physically organized/stored (the graph with layers)
- **Algorithm** — the steps used to build that structure and search through it
- **Vector index** — the end result: a data structure built by an algorithm, purpose-built for fast vector search

HNSW is all three simultaneously — it's an **algorithm** that **builds** a **graph-based data structure** that serves as a **vector index**. Calling it any one of those three isn't wrong, just incomplete.

**Note that You never interact with HNSW directly** — ChromaDB just uses it under the hood whenever you call `.query()`.

**If by default HNSW/Proximity is used then when does Clustering get used?**

In ChromaDB specifically, **HNSW (proximity graph) is what's always used** — you don't pick between clustering vs proximity as a user-facing choice. 

Clustering is more of a conceptual explanation ("similar vectors live near each other") rather than a separate algorithm ChromaDB offers.

#### Why Use HNSW?

* **Fast:** Avoids scanning the entire dataset.
* **Accurate:** Delivers near-exact results.
* **Scalable:** Handles millions to billions of vectors.
* **Versatile:** Works with various similarity metrics.

## Setting Up HNSW in Chroma DB

By default `ChromaDB` uses **HNSW automatically with sensible default parameters**, you don't touch anything. But if you want to tune it (change the distance metric, tweak speed/accuracy tradeoff, memory usage, etc.), you can configure it manually via the `hnsw` key in configuration during collection creation.

In Chroma DB, the HNSW vector index is configured during collection creation. Since HNSW is an approximate nearest neighbor algorithm, it's important to understand how its configuration affects both accuracy and performance. Parameters such as search speed, memory usage, and recall can all be tuned through the index settings. The example below demonstrates how to configure HNSW using the `hnsw` key:

In [1]:
# Setup
import chromadb
from chromadb.utils import embedding_functions

ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Collection creation
client = chromadb.Client()

collection = client.create_collection(
    name="my_collection_name",
    metadata={"topic": "query testing"},
    
    configuration={
        "hnsw": {
            "space": "cosine",   # Here we switched the distance metric from the default (L2/Euclidean) to cosine — which is the better choice for text/RAG work. That's the one parameter you'll actually use in practice.
            "ef_search": 100,
            "ef_construction": 100,
            "max_neighbors": 16
        },
        "embedding_function": ef
    }
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### HNSW Configuration Parameters

You'll rarely need to touch these — the defaults work fine for most cases. The only one you'll commonly change is `space`. Rest is for production-level tuning(with millions of vectors).

#### `space` — Distance Metric

Which similarity metric to use for search:

| Value | Metric | Use when |
|---|---|---|
| `l2` | Euclidean distance | Spatial/geometric data (default) |
| `ip` | Dot product | Magnitude matters (recommendation systems) |
| `cosine` | Cosine distance | **Text/RAG work — change to this** |

#### `ef_search` — Search Quality (default: 100)

When you run a query, HNSW doesn't just instantly return the top results — it first collects a **shortlist** of potentially relevant vectors to evaluate, then picks the best ones from that shortlist.

`ef_search` controls **how big that shortlist is**. Bigger shortlist = more vectors evaluated = more accurate results, but slower.
Think of it as: higher = looks at more options before deciding = more accurate results, but slower search and increased computatuonal cost.

#### `ef_construction` — Index Build Quality (default: 100)

Same idea but at **index build time** (when you `add()` documents). Higher = builds a better quality index = more accurate searches later, but takes longer and uses more memory when adding documents.

#### `max_neighbors` — Graph Density (default: 16)

How many "neighbor connections" each vector gets in the graph. Higher = more connections = easier to navigate the graph = better search accuracy, but more memory used and slower index building.

---

We can categorize the performance-based parameters into two types:

* `ef_search` directly controls the breadth of the search at query time, making it the most direct lever for search quality (recall) vs. query speed.

* `ef_construction` and `max_neighbors` affect the quality of the built index. A higher-quality, denser index (achieved with higher `ef_construction` and `max_neighbors`) provides a better foundation for searches, potentially leading to better accuracy. However, this quality comes at the cost of significantly longer index build times and higher memory consumption during construction and for storing the index.

## Performing Similarity Searches in Chroma DB

### Add data

Before performing a similarity search, we must add data to our collection:

In [2]:
collection.add(
    documents=[
        "Giant pandas are a bear species that lives in mountainous areas.",
        "A pandas DataFrame stores two-dimensional, tabular data",
        "I think everyone agrees that pandas are some of the cutest animals on the planet",
        "A direct comparison between pandas and polars indicates that polars is a more efficient library than pandas.",
    ],
    metadatas=[
        {"topic": "animals"},
        {"topic": "data analysis"},
        {"topic": "animals"},
        {"topic": "data analysis"},
    ],
    ids=["id1", "id2", "id3", "id4"]
)

As shown in the code above, all four documents mention "pandas." However, documents `id1` and `id3` refer to pandas as animals, while `id2` and `id4` refer to the Python library. Although the word "pandas" appears in every document, its meaning differs. This dataset is useful for evaluating whether a semantic search system can distinguish between these different meanings based on context.

### Querying in Chroma DB

**What is a Query?**

"query" is a general term that means different things in different contexts.

**In the LLM context** — a query = the question/prompt you send to the LLM to get a response.

**In the database context** — a query = a request you send to a database to retrieve data. This is the older, more general meaning — SQL has `SELECT` queries, search engines have search queries, etc.

**In ChromaDB specifically** — `collection.query()` means: *"search this collection and return the most similar documents to my search/query text."* You're querying the vector database, not an LLM.

Once a collection is created and populated, performing a similarity search is as simple as querying the database. In Chroma DB, the results are returned using an approximate nearest neighbor search powered by the HNSW algorithm, based on the distance metric specified during collection creation.

Let's query our collection using the query `cats`:

In [3]:
collection.query(
    query_texts=["cats"],
    n_results=10,
)

{'ids': [['id3', 'id1', 'id2', 'id4']],
 'embeddings': None,
 'documents': [['I think everyone agrees that pandas are some of the cutest animals on the planet',
   'Giant pandas are a bear species that lives in mountainous areas.',
   'A pandas DataFrame stores two-dimensional, tabular data',
   'A direct comparison between pandas and polars indicates that polars is a more efficient library than pandas.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'topic': 'animals'},
   {'topic': 'animals'},
   {'topic': 'data analysis'},
   {'topic': 'data analysis'}]],
 'distances': [[0.738014280796051,
   0.8351750373840332,
   0.8634339570999146,
   0.9299634695053101]]}

so here `query_texts=["cats"]` is your **search text**. You're saying *"find me the documents in this collection that are most semantically similar to the word 'cats'."*

ChromaDB then automatically embeds `"cats"` using the embedding function you set up, compares that vector against all stored document vectors, and returns the top `n_results` closest matches.

Note that querying the database involves passing the query, in a list, to the `query_texts` parameter in the `.query()` method. The optional parameter `n_results` controls the number of results to retrieve. In this case, `n_results` was set to 10, which exceeds the number of documents in the collection. Consequently, in this case, all results will be retrieved, ranked from most similar to the query to the least similar.

Also Note that, as expected, all four results got retrieved. However, the top two retrieved texts, those that resulted in the lowest cosine distance to the search query, were:

* "I think everyone agrees that pandas are some of the cutest animals on the planet" and

* "Giant pandas are a bear species that lives in mountainous areas."

Obviously, these two documents were retrieved because they relate to an animal as opposed to a programming library, and the query (cats) absent any other context likely refers to cats as animals as well. Moreover, the top search result was a document that discussed the cuteness of pandas, and, since cats are generally considered to be cute, likely resulted in that document being chosen over the more general one about pandas that referred to them living in mountainous areas.

These two documents were likely retrieved because they relate to animals rather than the Python programming library. Given that the query was simply "cats", with no additional context, it's reasonable to assume it refers to cats as animals. Moreover, the top retrieved document focused on the cuteness of pandas, which likely aligned more closely with the query "cats", a word also associated with cuteness, than a more general document about pandas' natural habitat.

## Querying with filters

Suppose we wanted to find the document that aligns most closely with the query polar bear using the following code:

In [6]:
collection.query(
    query_texts=["polar bear"],
    n_results=1,
)

{'ids': [['id4']],
 'embeddings': None,
 'documents': [['A direct comparison between pandas and polars indicates that polars is a more efficient library than pandas.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'topic': 'data analysis'}]],
 'distances': [[0.624370276927948]]}

**What went wrong?**

In this case, the word `polar` in the query was mistakenly matched with the term `polars`, referring to the Python data processing library mentioned in the document. As a result, the semantic search failed to retrieve documents about polar bears.

There are several ways to address this issue. One option is to refine the query by adding more context. Another is to try a different embedding model that may better capture the intended meaning. However, a simpler and more effective solution might be to apply **filters** to narrow the search results to a subset of the documents.

Consider the following code that enhances our original query by adding a metadata filter that narrows the search to documents about a specific topic:

In [7]:
collection.query(
    query_texts=["polar bear"],
    n_results=1,
    where={'topic': 'animals'}
)

{'ids': [['id1']],
 'embeddings': None,
 'documents': [['Giant pandas are a bear species that lives in mountainous areas.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'topic': 'animals'}]],
 'distances': [[0.7096825838088989]]}

The updated query now returns the correct result—a document about bears, rather than one related to Python libraries:

Alternatively, instead of using a metadata filter, we could perform a full-text search to include or exclude documents based on specific words or phrases. For example, the following query excludes all documents that contain the word "library":

In [8]:
collection.query(
    query_texts=["polar bear"],
    n_results=1,
    where_document={'$not_contains': 'library'}
)

{'ids': [['id1']],
 'embeddings': None,
 'documents': [['Giant pandas are a bear species that lives in mountainous areas.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'topic': 'animals'}]],
 'distances': [[0.7096825838088989]]}

Finally, it should be noted that there is nothing wrong with combining metadata filtering and full text search in Chroma DB:

In [ ]:
results = collection.query(
    query_texts=["polar bear"],
    n_results=1,
    where={'topic': 'animals'},
    where_document={'$not_contains': 'library'}
)
results

{'ids': [['id1']],
 'embeddings': None,
 'documents': [['Giant pandas are a bear species that lives in mountainous areas.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'topic': 'animals'}]],
 'distances': [[0.7096825838088989]]}